## Data Uploading to Google Drive

This notebook handles the automated uploading of cleaned data files from the local project directory to Google Drive.

### Process Overview:
1. Authenticate with Google Drive using PyDrive
2. Define list of cleaned data files to upload
3. For each file:
   - Check if file already exists in the interim folder
   - If exists: update the existing file
   - If not exists: create new file in the folder
4. Print success messages for each uploaded/updated file

### Files Uploaded:
- `../data/interim/renewal_calls_cleaned.csv`
- `../data/interim/emails_cleaned.csv`
- `../data/interim/cc_calls_cleaned.csv`

### Requirements:
- Google Drive API credentials (client_secrets.json)
- Environment variables: CLIENT_SECRETS_PATH, DRIVE_INTERIM_FOLDER_ID
- .env file with the required variables

In [ ]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import os
from dotenv import load_dotenv
load_dotenv()

gauth = GoogleAuth()
client_secrets_path = os.getenv("CLIENT_SECRETS_PATH")
gauth.LoadClientConfigFile(client_secrets_path)
gauth.LocalWebserverAuth()

drive = GoogleDrive(gauth)

files_to_upload = [
    '../data/interim/renewal_calls_cleaned.csv',
    '../data/interim/emails_cleaned.csv',
    '../data/interim/cc_calls_cleaned.csv',
    '../data/interim/billings_cleaned.csv'
]

folder_id = os.getenv("DRIVE_INTERIM_FOLDER_ID")

for file_path in files_to_upload:
    file_name = file_path.split('/')[-1]
    
    # Search if file already exists in the folder
    query = f"title='{file_name}' and '{folder_id}' in parents and trashed=false"
    file_list = drive.ListFile({'q': query}).GetList()
    
    if file_list:
        # File exists → update it
        file = file_list[0]
        file.SetContentFile(file_path)
        file.Upload()
        print(f"{file_name} updated successfully!")
    else:
        # File does not exist → create new
        file = drive.CreateFile({
            'title': file_name,
            'parents': [{'id': folder_id}]
        })
        file.SetContentFile(file_path)
        file.Upload()
        print(f"{file_name} uploaded successfully!")